# Workshop 2 - Power BI Dataset Readiness (solution)

![Power BI mockup](../../assets/images/powerbi_report_mockup.png)

## Scenario

RetailHub is ready to hand the Gold model to Power BI. This solution shows a complete BI handoff decision: report sources, Import vs DirectQuery reasoning, incremental-refresh validation, BI contract and cost guardrails.

## Objectives

After reviewing this solution you will be able to:

- defend the selected summary and drill-through sources,
- explain the dataset connection-mode decision,
- validate a half-open incremental-refresh window,
- complete the BI contract with concrete values,
- define practical SQL Warehouse cost guardrails.

## Prerequisites

Before running, complete:

1. `setup/00_setup.ipynb`
2. `notebooks/demo/day1_03_gold_modeling_for_powerbi.ipynb`
3. `notebooks/demo/day2_01_powerbi_semantic_model.ipynb`

## Setup

Initialize the environment, resolve the participant catalog and expose the shared variables used by this workshop.

In [ ]:
%run ../../setup/00_setup

### Configuration

Display the active RetailHub catalog context and validate prerequisite objects before starting the tasks.

In [ ]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))

### Runtime Pre-check

Fail fast if an upstream demo notebook has not created the required Gold objects.

In [ ]:
required_tables = [
    f"{GOLD}.fact_sales_dashboard_monthly",
    f"{GOLD}.v_fact_sales_incremental",
]

missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    print("[BLOCKED] Missing required tables:")
    for t in missing:
        print("  -", t)
    print()
    print("Run this notebook first: notebooks/demo/day1_03_gold_modeling_for_powerbi.ipynb (for fact_sales_dashboard_monthly) and notebooks/demo/day2_01_powerbi_semantic_model.ipynb (for v_fact_sales_incremental)")
    raise Exception("Pre-check failed: missing prerequisite tables. Run \"notebooks/demo/day1_03_gold_modeling_for_powerbi.ipynb (for fact_sales_dashboard_monthly) and notebooks/demo/day2_01_powerbi_semantic_model.ipynb (for v_fact_sales_incremental)\" first.")

print("[OK] Pre-check passed, all required tables exist:")
for t in required_tables:
    print("  -", t)

## Success Criteria

You are done when:

1. Summary-page source is chosen and justified (row count, grain known).
2. Drill-through source is chosen and justified (date boundaries known).
3. Import vs DirectQuery is decided for THIS dataset, with reasoning - not
   just copied from the Day 2 demo.
4. You ran your own boundary test against `gold.v_fact_sales_incremental`
   and it returns only the requested half-open window.
5. The BI contract (`docs/templates/bi-contract.md` shape) is filled for
   this workshop's choices.
6. The cost guardrails checklist (`docs/templates/cost-awareness-checklist.md`
   shape) is filled for this report.

## Tasks

1. Wybierz source dla summary page.
2. Wybierz source dla drill-through.
3. Zdefiniuj Import vs DirectQuery dla tego datasetu.
4. Zweryfikuj incremental refresh (twoj wlasny boundary test).
5. Wypelnij BI contract.
6. Wypelnij cost guardrails.

## Task 1 - wybierz source dla summary page

The executive summary page needs month-level KPI cards and a trend line -
both are aggregate questions, so the monthly Gold table is enough and is
far cheaper to scan than line-grain detail.

In [ ]:
summary_profile = spark.sql(f"""
SELECT
  MIN(year_month) AS min_month,
  MAX(year_month) AS max_month,
  COUNT(*) AS aggregate_rows,
  SUM(revenue) AS revenue
FROM {GOLD}.fact_sales_dashboard_monthly
""")
summary_profile.show()

print("Decision: summary page source = gold.fact_sales_dashboard_monthly")
print("Reason: KPI cards and trend lines are month-grain questions; the")
print("monthly aggregate answers them without scanning line-grain rows.")

## Task 2 - wybierz source dla drill-through

The drill-through page must show individual order lines behind any KPI
card, so it needs line grain with real date boundaries - that is exactly
what `v_fact_sales_incremental` provides (24-month rolling window).

In [ ]:
detail_profile = spark.sql(f"""
SELECT
  MIN(order_date) AS min_date,
  MAX(order_date) AS max_date,
  COUNT(*) AS detail_rows
FROM {GOLD}.v_fact_sales_incremental
""")
detail_profile.show()

print("Decision: drill-through source = gold.v_fact_sales_incremental")
print("Reason: drill-through needs order-line grain; this view is the only")
print("Gold object built at that grain with a bounded, indexable date range.")

## Task 3 - zdefiniuj Import vs DirectQuery (ten dataset)

Apply the Day 2 semantic model decision table to this report specifically:

| Question | Answer for this report |
|---|---|
| Does any visual need sub-hour freshness? | No - sales close end-of-day, daily Gold refresh is enough |
| Is data volume small enough for Import? | Yes - monthly aggregate is a few thousand rows, incremental view is a bounded 24-month window |
| Are there concurrent users hitting the same visuals? | Yes (workshop/demo audience) - DirectQuery cost scales with `visuals x filter changes x users`, Import does not |
| Is there a genuine operational/live use case? | Only a small "today's orders" operational page, if one is added later |

**Decision for this dataset: Import is the baseline for both the summary
page and the drill-through page.** DirectQuery is reserved as the
exception, only if a future operational page needs intra-day freshness -
and even then it must point at a Gold aggregate, never at Silver detail.

## Task 4 - zweryfikuj incremental refresh (twoj test)

Run a boundary test against `gold.v_fact_sales_incremental` using a
different window than the Day 2 demo example, to prove the half-open contract
holds in general, not just for one hard-coded range.

In [ ]:
range_start = "2025-04-01"
range_end = "2025-07-01"

window_check = spark.sql(f"""
SELECT
  COUNT(*) AS rows_in_window,
  MIN(order_date) AS min_date,
  MAX(order_date) AS max_date
FROM {GOLD}.v_fact_sales_incremental
WHERE order_date >= DATE '{range_start}'
  AND order_date <  DATE '{range_end}'
""").first()

print(window_check)
assert str(window_check["min_date"]) >= range_start, "Window starts before RangeStart"
assert str(window_check["max_date"]) < range_end, "Window includes the RangeEnd boundary"

boundary_row = spark.sql(f"""
SELECT COUNT(*) AS rows_on_range_end
FROM {GOLD}.v_fact_sales_incremental
WHERE order_date = DATE '{range_end}'
""").first()
print("Rows exactly on RangeEnd (must be excluded from the window above):", boundary_row["rows_on_range_end"])

print("[OK] half-open RangeStart/RangeEnd predicate verified for a second window")

## Task 5 - wypelnij BI contract

Filled `docs/templates/bi-contract.md` for this workshop's dataset:

| Area | Decision |
|---|---|
| Summary source | `gold.fact_sales_dashboard_monthly` |
| Detail/drill-through source | `gold.v_fact_sales_incremental` |
| Grain (summary) | one row per `year_month` x region x category x channel |
| Grain (detail) | one row per order line, 24-month rolling window |
| Baseline mode | Import |
| Live option | DirectQuery only for a future small operational page on a Gold aggregate, never on Silver |
| Refresh pattern | incremental by `order_date`, half-open `RangeStart`/`RangeEnd` window |
| Cost guardrail | no visual may query Silver detail directly |
| Refresh owner | Data team |
| Business owner | Sales Ops |
| Technical owner | BI team |

## Task 6 - wypelnij cost guardrails

Filled `docs/templates/cost-awareness-checklist.md` for this report:

| Area | Question | Decision |
|---|---|---|
| Warehouse size | What size is enough for the demo/report? | Small/X-Small Serverless SQL Warehouse - aggregate-grain queries, low concurrency in a workshop setting |
| Auto-stop | How long should idle compute stay warm? | Auto-stop at 5-10 minutes; refresh runs are short and infrequent |
| Import mode | Can refresh be scheduled instead of live queries? | Yes - daily scheduled Import refresh after the Gold job completes |
| DirectQuery | Which visuals can trigger expensive SQL? | None today; if added later, restrict to a single operational page on a Gold aggregate |
| Aggregates | Which Gold aggregates reduce scan size? | `gold.fact_sales_dashboard_monthly` for every summary visual; `v_fact_sales_incremental` only behind drill-through |
| Monitoring | Where do we inspect query and billing usage? | SQL warehouse monitoring UI + system billing tables, reviewed weekly |

## Self-check

In [ ]:
assert spark.catalog.tableExists(f"{GOLD}.fact_sales_dashboard_monthly"), "Missing monthly aggregate"
assert spark.catalog.tableExists(f"{GOLD}.v_fact_sales_incremental"), "Missing incremental view"

summary_rows = spark.sql(f"SELECT COUNT(*) AS rows FROM {GOLD}.fact_sales_dashboard_monthly").first()["rows"]
detail_rows = spark.sql(f"SELECT COUNT(*) AS rows FROM {GOLD}.v_fact_sales_incremental").first()["rows"]
assert summary_rows > 0, "Summary aggregate is empty"
assert detail_rows > 0, "Incremental view is empty"

window = spark.sql(f"""
SELECT MIN(order_date) AS min_date, MAX(order_date) AS max_date
FROM {GOLD}.v_fact_sales_incremental
WHERE order_date >= DATE '2025-04-01'
  AND order_date <  DATE '2025-07-01'
""").first()
assert str(window["min_date"]) >= "2025-04-01", "Window starts too early"
assert str(window["max_date"]) < "2025-07-01", "Window includes RangeEnd boundary"

print("[OK] Workshop 2 self-check passed: Import baseline, DirectQuery exception only,")
print("[OK] BI contract and cost guardrails filled, incremental window verified")

## Summary

This solution demonstrates a complete Power BI dataset readiness handoff: selected Gold objects, connection-mode reasoning, incremental-refresh validation, BI contract and cost guardrails.

Use it to compare participant decisions against a practical BI product standard.